# Section 2 -- Inspecting an A2A agent

**Before you run anything:** the A2A server must already be up.

```bash
./scripts/run_a2a_server.sh
```

MCP connected an agent to *tools*. A2A connects an agent to *another agent*.

The difference is not cosmetic. A tool is a function: you call it, it returns.
A peer agent is an autonomous thing with its own model, its own instructions,
and its own tools. You do not call it -- you delegate to it.

In [ ]:
import json
import httpx

BASE = "http://localhost:8001/a2a/billing_agent"

# a2a-sdk 0.3.x served the card at /.well-known/agent.json; 1.x serves it at
# /.well-known/agent-card.json. In real code use the ADK constant
# AGENT_CARD_WELL_KNOWN_PATH rather than hardcoding either.
CARD_URL = f"{BASE}/.well-known/agent-card.json"
print(CARD_URL)

## 1. The agent card

This is all of A2A discovery. A JSON document at a well-known URL.

No registry, no service mesh, no broker. If you can GET this file, you can work
with the agent.

In [ ]:
card = httpx.get(CARD_URL).json()
print(json.dumps(card, indent=2))

## 2. Reading the card the way a caller does

Three fields carry the weight:

- **`skills`** -- what it can do, in natural language. This is what a calling
  agent's model reads to decide whether to delegate here. Same lesson as MCP
  docstrings: vague description, no delegation.
- **`capabilities`** -- protocol-level features, e.g. streaming.
- **`url`** -- where to actually send work.

In [ ]:
# An A2A agent card is served in TWO shapes at once, merged into one document.
#
#   legacy (protocol 0.3):  a top-level "url" plus "preferredTransport"
#   current (protocol 1.0): a "supportedInterfaces" list, each entry carrying
#                           its own url + protocolBinding + protocolVersion
#
# The server emits both so that old and new clients can each find what they
# expect. Read the modern field first and fall back -- that is what a real
# client library does for you.

def resolve_rpc_url(card: dict) -> str:
    interfaces = card.get("supportedInterfaces") or []
    if interfaces and interfaces[0].get("url"):
        return interfaces[0]["url"]
    return card["url"]  # legacy 0.3 placement


RPC_URL = resolve_rpc_url(card)

print("name:  ", card["name"])
print("rpc:   ", RPC_URL)
print("caps:  ", card.get("capabilities"))
print()
for skill in card.get("skills", []):
    print(f"- {skill['id']}: {skill['description']}")
    print(f"  tags: {', '.join(skill.get('tags', []))}")


> **Watch the `url` field.** It is absolute. That is why this workshop keeps
> all traffic inside the Codespace -- route it through a public forwarded URL
> and the card will still advertise `localhost`, so discovery succeeds and
> every call fails. This is a genuine deployment gotcha, not a workshop
> limitation.

## 3. Sending work over the wire

A2A is JSON-RPC too. Here is a delegation with no framework involved -- exactly
what `triage_agent` sends when it hands a billing question to `billing_agent`.

In [ ]:
import uuid

# The A2A 0.3 JSON-RPC dialect. Note there is no bespoke "billing" verb here --
# `message/send` is the same call you would make to ANY A2A agent. The agent's
# specialty lives in its card, not in its method names. That uniformity is what
# makes agents composable.
payload = {
    "jsonrpc": "2.0",
    "id": str(uuid.uuid4()),
    "method": "message/send",
    "params": {
        "message": {
            "role": "user",
            "messageId": str(uuid.uuid4()),
            "parts": [
                {"kind": "text", "text": "What is the refund policy for annual plans?"}
            ],
        }
    },
}

response = httpx.post(RPC_URL, json=payload, timeout=120.0)
print("status:", response.status_code)
print(json.dumps(response.json(), indent=2)[:1500])


## 4. Now watch an agent do it

Start the dev UI in a second terminal:

```bash
./scripts/run_web.sh
```

Open http://127.0.0.1:8002, pick **triage_agent**, and ask:

> *Why was I charged twice this month?*

Watch the trace. Triage decides this is billing, and hands off to
`billing_agent` -- which is a different process, reached over HTTP, that triage
knows about only through the card you read above.

Then open `src/helpdesk/a2a/local/triage_agent/agent.py`. The remote agent is
declared in three lines and then dropped into `sub_agents=[...]` beside local
ones. From the model's point of view there is no difference between a local
sub-agent and one running in another process.

That is the abstraction A2A buys you.

---

Next: `docs/03-lab-interop.md`.